In [1]:
!pip install streamlit langchain-google-genai pyngrok geopy streamlit-js-eval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 94.5 MB/s eta 0:00:00


In [4]:
%%writefile therapist_app.py
import streamlit as st
import os
import json
import time
from datetime import datetime
from typing import TypedDict, List, Optional
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# --- 1. PERSISTENCE (Same as before) ---
def load_patient_file(patient_id):
    if os.path.exists("clinical_records.json"):
        with open("clinical_records.json", "r") as f:
            data = json.load(f)
            return data.get(patient_id, {"sessions": [], "profile": {"care_path": "Pending"}})
    return {"sessions": [], "profile": {"care_path": "Pending"}}

def save_clinical_note(patient_id, session_note):
    data = {}
    if os.path.exists("clinical_records.json"):
        with open("clinical_records.json", "r") as f:
            data = json.load(f)
    if patient_id not in data:
        data[patient_id] = {"sessions": [], "profile": {"care_path": "Pending"}}
    data[patient_id]["sessions"].append({"date": str(datetime.now().date()), "note": session_note})
    with open("clinical_records.json", "w") as f:
        json.dump(data, f, indent=4)

# --- 2. DYNAMIC SESSION NODE ---

def get_therapist_response(patient_id, user_input=None):
    file = load_patient_file(patient_id)
    session_count = len(file["sessions"]) + 1
    care_path = file["profile"]["care_path"]

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        google_api_key=os.environ.get("GEMINI_API_KEY")
    )

    # Context Setting
    if session_count == 1:
        context = "This is the INTAKE SESSION. We are gathering history, symptoms, and background."
    else:
        last_note = file["sessions"][-1]["note"]
        context = f"This is a FOLLOW-UP session. Care Path: {care_path}. Last session summary: {last_note}"

    # Logic: If user_input is None, we are STARTING the session and need a guided question.
    if not user_input:
        prompt = f"{context}\nTask: Generate a warm, professional opening question to start this session."
    else:
        prompt = f"{context}\nPatient said: {user_input}\nTask: Respond with empathy and ask a deep follow-up question."

    for attempt in range(3):
        try:
            res = llm.invoke(prompt)
            return res.content
        except:
            time.sleep(2)
    return "I'm reflecting on our session. Please go on."

# --- 3. STREAMLIT UI ---

st.set_page_config(page_title="AI Therapist", layout="wide")
st.title("🧘 Mental Wellness Partner")

patient_id = st.sidebar.text_input("Patient ID", "User_Alpha")
patient_data = load_patient_file(patient_id)

# Sidebar History
st.sidebar.subheader("📋 Past Session Notes")
for s in reversed(patient_data["sessions"]):
    st.sidebar.info(f"📅 {s['date']}\n{s['note']}")

# Session State for Chat
if "messages" not in st.session_state:
    st.session_state.messages = []
    # Trigger the first dynamic question automatically
    with st.spinner("Preparing session..."):
        initial_q = get_therapist_response(patient_id)
        st.session_state.messages.append({"role": "assistant", "content": initial_q})

# Display Chat
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# User Input
if prompt := st.chat_input("Type your response here..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        response = get_therapist_response(patient_id, prompt)
        st.markdown(response)
        st.session_state.messages.append({"role": "assistant", "content": response})

# Session Management
if st.button("💾 End Session & Save Clinical Note"):
    # Summarize the whole chat for the note
    summary_llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=os.environ.get("GEMINI_API_KEY"))
    chat_history = "\n".join([f"{m['role']}: {m['content']}" for m in st.session_state.messages])
    summary_res = summary_llm.invoke(f"Summarize this therapy session into a professional clinical note:\n{chat_history}")

    save_clinical_note(patient_id, summary_res.content)
    st.success("Clinical record updated. You can now close the tab.")

Overwriting therapist_app.py


In [5]:
from pyngrok import ngrok
from google.colab import userdata
import os

# 1. Terminate any existing tunnels to avoid 'address already in use' errors
ngrok.kill()

# 2. Retrieve your NGROK_AUTH token from Colab Secrets (the 🔑 icon)
# Make sure you have added a secret named 'NGROK_AUTH'
try:
    auth_token = userdata.get('NGROK_AUTH')
    ngrok.set_auth_token(auth_token)
except Exception as e:
    print("❌ Error: NGROK_AUTH secret not found. Check your Colab Secrets!")

# 3. Inject AI keys into the environment for the Streamlit process
# This ensures 'therapist_app.py' can access them via os.environ
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

# 4. Open a tunnel on port 8501 (the default Streamlit port)
public_url = ngrok.connect(8501).public_url
print(f"✅ THERAPIST AI IS LIVE!")
print(f"🔗 Click here to start the session: {public_url}")

# 5. Run the app
!streamlit run therapist_app.py --server.port 8501

✅ THERAPIST AI IS LIVE!
🔗 Click here to start the session: https://cocciferous-dioicous-gino.ngrok-free.dev


2026-05-12 09:02:04.867 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.24.174.73:8501

  Stopping...
  Stopping...
  Stopping...
  Stopping...
^C
